In [1]:
# build_chip.py -- builds the die using positions from pad_map.py
from functools import partial
from pathlib import Path

import gdsfactory as gf
import pad_map as pm

gf.gpdk.PDK.activate()

HEATER_LENGTHS = [200, 320, 500, 600]  # MZI 1..4, one per pad column (left to right)
LOOPBACK_X_UM = 1250               # loopback center, left of the first pad column
DEVICE_ROW_Y_UM = 450              # bottom edge of the device row
CHIP_LABEL = "IK v1"               # your initials + version
DROP_UM = 50                       # how far routes drop below the heater terminals
SIDE_CLEARANCE_UM = 100            # LO route: gap from device right edge to riser
TOP_CHANNEL_GAP_UM = 100           # LO route: gap below the top pads to the channel

assert len(HEATER_LENGTHS) == pm.N_HEATERS, "HEATER_LENGTHS and pad_map.N_HEATERS disagree"


@gf.cell
def mzi_heater(heater_length: float = 320.0, delta_length: float = 40.0):
    heater = partial(gf.components.straight_heater_metal, length=heater_length)
    return gf.components.mzi(
        delta_length=delta_length,
        length_x=heater_length,
        straight_x_top=heater,
    )


@gf.cell
def loopback():
    return gf.routing.add_fiber_array(gf.components.straight(length=20))


def heater_terminals(ref):
    """Leftmost and rightmost south-facing electrical ports (after rotation)."""
    ports = [
        p for p in ref.ports
        if p.port_type == "electrical" and round(p.orientation) % 360 == 270
    ]
    if len(ports) < 2:
        raise ValueError(f"No heater ports found. Ports: {[p.name for p in ref.ports]}")
    ports.sort(key=lambda p: p.x)
    return ports[0], ports[-1]


def add_segment(c, x0, y0, x1, y1, w, layer="M3"):
    """Manhattan metal segment of width w, extended w/2 past each end so corners fill."""
    h = w / 2
    xa, xb = min(x0, x1) - h, max(x0, x1) + h
    ya, yb = min(y0, y1) - h, max(y0, y1) + h
    c.add_polygon([(xa, ya), (xb, ya), (xb, yb), (xa, yb)], layer=layer)


def add_path(c, points, w, layer="M3"):
    """Chain of Manhattan segments through the given (x, y) points."""
    for (x0, y0), (x1, y1) in zip(points, points[1:]):
        add_segment(c, x0, y0, x1, y1, w, layer)


@gf.cell
def chip():
    c = gf.Component()

    # 1. Die outline: lower-left corner at (0, 0), matching pad_map's origin
    c << gf.components.rectangle(size=(pm.DIE_W_UM, pm.DIE_H_UM), layer="FLOORPLAN")

    # 2. Pads from the pad map, each labeled with its number (on the die-edge side)
    pads = {}
    for p in pm.PADS:
        ref = c << gf.components.pad(size=(pm.PAD_SIZE_UM, pm.PAD_SIZE_UM))
        ref.x = p["x_um"]
        ref.y = p["y_um"]
        pads[p["net"]] = p

        t = c << gf.components.text(str(p["num"]), size=30, layer="TEXT")
        t.x = p["x_um"]
        if p["side"] == "bottom":
            t.ymax = p["y_um"] - pm.PAD_SIZE_UM / 2 - 5  # just below the pad
        else:
            t.ymin = p["y_um"] + pm.PAD_SIZE_UM / 2 + 5  # just above the pad

    bottom_pad_top = pm.PAD_ROW_Y_BOTTOM_UM + pm.PAD_SIZE_UM / 2
    top_pad_bottom = pm.PAD_ROW_Y_TOP_UM - pm.PAD_SIZE_UM / 2
    y_top_channel = top_pad_bottom - TOP_CHANNEL_GAP_UM

    # 3. Loopback, left of the first pad column
    lb = c << loopback()
    lb.rotate(180)
    lb.x = LOOPBACK_X_UM
    lb.ymin = DEVICE_ROW_Y_UM
    t = c << gf.components.text("LOOPBACK", size=20, layer="TEXT")
    t.x = lb.x
    t.ymin = lb.ymax + 10

    # 4. One MZI per pad column, centered on it; HI -> bottom pad, LO -> top pad
    for k, L in enumerate(HEATER_LENGTHS, start=1):
        dev = gf.routing.add_fiber_array(
            mzi_heater(heater_length=L), with_loopback=False, with_loopback_inside=False
        )
        ref = c << dev
        ref.rotate(180)          # grating couplers up, heater down
        ref.x = pm.COLUMN_X_UM[k - 1]
        ref.ymin = DEVICE_ROW_Y_UM

        t = c << gf.components.text(f"MZI{k}_H{L:g}", size=20, layer="TEXT")
        t.x = ref.x
        t.ymin = ref.ymax + 10
        if t.ymax + 30 > y_top_channel:
            raise ValueError("Top route channel runs into the device labels")

        hi, lo = heater_terminals(ref)
        hi_pad, lo_pad = pads[f"HTR{k}_HI"], pads[f"HTR{k}_LO"]
        w = hi.width
        y_drop = min(hi.y, lo.y, ref.ymin) - DROP_UM   # channel just below the heater
        if y_drop - 30 < bottom_pad_top:
            raise ValueError("Not enough room between the heater and the bottom pads")

        # HI: down, across to the column center, down into the bottom pad
        add_path(c, [
            (hi.x, hi.y),
            (hi.x, y_drop),
            (hi_pad["x_um"], y_drop),
            (hi_pad["x_um"], bottom_pad_top - w / 2),
        ], w)

        # LO: down, right past the device, up beside it, back over the top, up into the top pad
        x_riser = ref.xmax + SIDE_CLEARANCE_UM
        add_path(c, [
            (lo.x, lo.y),
            (lo.x, y_drop),
            (x_riser, y_drop),
            (x_riser, y_top_channel),
            (lo_pad["x_um"], y_top_channel),
            (lo_pad["x_um"], top_pad_bottom + w / 2),
        ], w)

    # 5. Chip label, top-left corner
    t = c << gf.components.text(CHIP_LABEL, size=40, layer="TEXT")
    t.xmin = 30
    t.ymax = pm.DIE_H_UM - 30

    return c


if __name__ == "__main__":
    assert pm.check(), "Fix pad_map.py first"
    c = chip()

    # Anything outside the die outline makes the bounding box bigger than the die
    tol = 0.01
    if (c.xmin < -tol or c.ymin < -tol
            or c.xmax > pm.DIE_W_UM + tol or c.ymax > pm.DIE_H_UM + tol):
        print(f"WARNING: layout extends past the die: "
              f"x {c.xmin:.0f}..{c.xmax:.0f}, y {c.ymin:.0f}..{c.ymax:.0f} um")
    else:
        print("Layout fits inside the die.")

    c.show()
    Path("chip").mkdir(exist_ok=True)
    c.write_gds("chip/mzi_die.gds")

Layout fits inside the die.
